In [54]:
import pandas as pd
from pythonTool import *
import sys
sys.path.append('\\\\htisec.local\\HK Data\\Department\\PB\\PB\\BCTMate')
from bctmate.price.price_func import *

# 获取相关数据

In [224]:
# 获取最新股票信息
price_df = load_daily_price_by_date('20250825')
code_name_mapping = price_df.set_index('标的代码')['标的名称'].to_dict()
multiplier_mapping = price_df.set_index('标的代码')['标的乘数'].to_dict()
market_mapping = price_df.set_index('标的代码')['市场'].to_dict()

In [247]:
# 获取月初股票价格
price_df = load_daily_price_by_date('20250801')
exposure_price_mapping = price_df.set_index('标的代码')['敞口计算价格'].to_dict()
index_price = price_df.set_index('标的代码').loc['SHSZ300 Index', '敞口计算价格']
other_price_mapping = {'IFBZ5 Index': index_price}
exposure_price_mapping.update(other_price_mapping)


# 获取当前cad头寸
sql = 'select 标的代码, 标的名称, 分红调整剩余数量, 客户头寸方向, 标的类型 from trade_detail where 估值日期=2025.08.25, 机构名称="HTI Financial Solutions Limited-CAD"'
cad_pos = ddb(db='bctmate').load_table_by_sql('dfs://bctmate_report', 'trade_detail', sql)
cad_pos = cad_pos.loc[cad_pos['客户头寸方向'] != 'PTH']

# 合并头寸
cad_pos['quantity'] = cad_pos['客户头寸方向'].apply(lambda x: 1 if x=='多头' else -1) * cad_pos['分红调整剩余数量']
cad_pos = cad_pos.groupby(['标的代码', '标的名称', '标的类型'])['quantity'].sum().reset_index()


# 获取指数成分股
index_components = ddb('percy').load_table_by_sql('dfs://jq_price', 'index_components', 'select * from index_components where date=2025.08.01')
index_components = index_components.loc[index_components['index_code'].isin(['000300.XSHG', '000905.XSHG', '000852.XSHG'])]
index_components['bbg code'] = index_components['code'].apply(lambda x: x.split('.')[0] + ' CH Equity')
index_code_mapping = {'000905.XSHG': 'SH000905 Index', '000300.XSHG':'SHSZ300 Index', '000852.XSHG':'CSI1000 Index'}
index_components['bbg index_code'] = index_components['index_code'].map(index_code_mapping)

# 补全头寸
all_code = list(set(index_components['bbg code']).union(set(cad_pos['标的代码'])))
cad_pos = cad_pos.set_index('标的代码').reindex(all_code).reset_index()
cad_pos['标的类型'] = cad_pos['标的类型'].fillna('股票')
cad_pos['quantity'] = cad_pos['quantity'].fillna(0)
cad_pos['标的名称'] = cad_pos['标的代码'].map(code_name_mapping)
cad_pos['multiplier'] = cad_pos['标的代码'].map(multiplier_mapping)

cad_pos['exposure_price'] = cad_pos['标的代码'].map(exposure_price_mapping)
cad_pos['market'] = cad_pos['标的代码'].map(market_mapping)

# 只保留A股
cad_pos = cad_pos.loc[cad_pos['market'] == 'A股']

index_mapping = index_components.set_index('bbg code')['bbg index_code'].to_dict()
weight_mapping = index_components.set_index('bbg code')['weight'].to_dict()

cad_pos['index'] = cad_pos['标的代码'].map(index_mapping)
cad_pos['weight'] = cad_pos['标的代码'].map(weight_mapping)

# 计算8月1日市值
cad_pos['market_value'] = cad_pos['quantity'] * cad_pos['exposure_price'] * cad_pos['multiplier'] 



In [249]:
# 确认有没有价格缺失
cad_pos.loc[cad_pos['index'].isna()]

,标的代码,标的名称,标的类型,quantity,multiplier,exposure_price,market,index,weight,market_value
215,300894 CH Equity,火星人,股票,1200.0,1.0,14.2400,A股,NaN,NaN,1.708800e+04
220,IFBU5 Index,IF2509,股指期货,-184.0,300.0,4054.9286,A股,NaN,NaN,-2.238321e+08
590,FFDU5 Index,IC2509,股指期货,-58.0,200.0,6213.2014,A股,NaN,NaN,-7.207314e+07
708,IFDU5 Index,IM2509,股指期货,-173.0,200.0,6670.4720,A股,NaN,NaN,-2.307983e+08
839,605365 CH Equity,立达信,股票,400.0,1.0,14.8500,A股,NaN,NaN,5.940000e+03
844,IFDH6 Index,IM2603,股指期货,-55.0,200.0,6670.4720,A股,NaN,NaN,-7.337519e+07
958,IFDZ5 Index,IM2512,股指期货,-60.0,200.0,6670.4720,A股,NaN,NaN,-8.004566e+07
1261,002812 CH Equity,恩捷股份,股票,1700.0,1.0,29.5600,A股,NaN,NaN,5.025200e+04
1363,FFDZ5 Index,IC2512,股指期货,-92.0,200.0,6213.2014,A股,NaN,NaN,-1.143229e+08
1562,000627 CH Equity,*ST天茂,股票,78100.0,1.0,1.4600,A股,NaN,NaN,1.140260e+05


In [250]:
cad_pos['客户头寸方向'] = cad_pos['quantity'].apply(lambda x: '多头' if x>=0 else '空头')
print('多空头市值')
notional_ls = cad_pos.groupby('客户头寸方向')['market_value'].sum()
print(notional_ls)

print('净敞口')
print(notional_ls['多头'] + notional_ls['空头'])


多空头市值
客户头寸方向
多头    8.675341e+08
空头   -8.674360e+08
Name: market_value, dtype: float64
净敞口
98127.5799999237


# 查看没有权重的头寸

In [251]:
pos_no_index = cad_pos.loc[cad_pos['weight'].isna()]
pos_no_index

,标的代码,标的名称,标的类型,quantity,multiplier,exposure_price,market,index,weight,market_value,客户头寸方向
215,300894 CH Equity,火星人,股票,1200.0,1.0,14.2400,A股,NaN,NaN,1.708800e+04,多头
220,IFBU5 Index,IF2509,股指期货,-184.0,300.0,4054.9286,A股,NaN,NaN,-2.238321e+08,空头
590,FFDU5 Index,IC2509,股指期货,-58.0,200.0,6213.2014,A股,NaN,NaN,-7.207314e+07,空头
708,IFDU5 Index,IM2509,股指期货,-173.0,200.0,6670.4720,A股,NaN,NaN,-2.307983e+08,空头
839,605365 CH Equity,立达信,股票,400.0,1.0,14.8500,A股,NaN,NaN,5.940000e+03,多头
844,IFDH6 Index,IM2603,股指期货,-55.0,200.0,6670.4720,A股,NaN,NaN,-7.337519e+07,空头
958,IFDZ5 Index,IM2512,股指期货,-60.0,200.0,6670.4720,A股,NaN,NaN,-8.004566e+07,空头
1261,002812 CH Equity,恩捷股份,股票,1700.0,1.0,29.5600,A股,NaN,NaN,5.025200e+04,多头
1363,FFDZ5 Index,IC2512,股指期货,-92.0,200.0,6213.2014,A股,NaN,NaN,-1.143229e+08,空头
1562,000627 CH Equity,*ST天茂,股票,78100.0,1.0,1.4600,A股,NaN,NaN,1.140260e+05,多头


In [252]:
pos_no_index.loc[pos_no_index['标的类型'] == '股票']['market_value'].sum()

281108.63

In [253]:
cad_pos['weight'] = cad_pos['weight'].fillna(0)

# 计算理论数量与差额

In [254]:
# 获取月初股票价格
price_tmp = load_daily_price_by_date('20250825')
future_underlying_mapping = price_tmp.set_index('标的代码')['挂钩标的'].to_dict()
# 计算指数notional
cad_pos_s = cad_pos.loc[cad_pos['客户头寸方向'] == '空头']
cad_pos_s['挂钩标的'] = cad_pos_s['标的代码'].map(future_underlying_mapping)
index_notional_mapping= (cad_pos_s.groupby('挂钩标的')['market_value'].sum() * -1).to_dict()

In [284]:
cad_pos_l = cad_pos.loc[cad_pos['客户头寸方向'] == '多头']
# 计算理论金额
cad_pos_l['index_notional'] = cad_pos_l['index'].map(index_notional_mapping)
cad_pos_l['underlying_theory_mv'] = cad_pos_l['index_notional'] * cad_pos_l['weight']/100
cad_pos_l['underlying_theory_quantity'] = cad_pos_l['underlying_theory_mv'] / cad_pos_l['exposure_price'] / cad_pos_l['multiplier']
cad_pos_l['amount_diff'] = cad_pos_l['underlying_theory_mv'] - cad_pos_l['market_value']
cad_pos_l['quantity_diff'] = cad_pos_l['underlying_theory_quantity'] - cad_pos_l['quantity']
cad_pos_l['amount_diff_abs'] = cad_pos_l['amount_diff'].abs()

# 挑选适合调整的头寸

In [285]:
pos_adjust = cad_pos_l.loc[cad_pos_l['amount_diff'].abs() > 0]
pos_adjust.groupby('index')['amount_diff'].sum().sum()

-343.3547696409805

In [294]:
# 看看各个指数多空差异, 主要是做多了1000, 做空了300 和 500
index_exposure = cad_pos_l.groupby('index')['market_value'].sum() - pd.Series(index_notional_mapping)
index_exposure

index
CSI1000 Index     734141.11
SH000905 Index   -811207.42
SHSZ300 Index    -105914.74
dtype: float64

In [312]:
# 处理300头寸
pos_adjust_300 = pos_adjust.loc[pos_adjust['index']== 'SHSZ300 Index']
pos_adjust_300 = pos_adjust_300.sort_values(by='amount_diff_abs', ascending=False)

pos_adjust_300['expanding_amount'] = pos_adjust_300['amount_diff'].expanding().sum()
pos_adjust_300['expanding_amount_adjust'] = pos_adjust_300['expanding_amount'] + index_exposure.loc['SHSZ300 Index']
pos_adjust_300['expanding_amount_adjust_abs'] = pos_adjust_300['expanding_amount_adjust'].abs()
pos_adjust_300 = pos_adjust_300.reset_index(drop=True)
pos_adjust_300.to_excel('300调整明细.xlsx')

In [313]:
# 处理500头寸
pos_adjust_500 = pos_adjust.loc[pos_adjust['index']== 'SH000905 Index']
pos_adjust_500 = pos_adjust_500.sort_values(by='amount_diff_abs', ascending=False)

pos_adjust_500['expanding_amount'] = pos_adjust_500['amount_diff'].expanding().sum()
pos_adjust_500['expanding_amount_adjust'] = pos_adjust_500['expanding_amount'] + index_exposure.loc['SH000905 Index']
pos_adjust_500['expanding_amount_adjust_abs'] = pos_adjust_500['expanding_amount_adjust'].abs()
pos_adjust_500 = pos_adjust_500.reset_index(drop=True)
pos_adjust_500.to_excel('500调整明细.xlsx')

In [314]:
# 处理1000头寸
pos_adjust_1000 = pos_adjust.loc[pos_adjust['index']== 'CSI1000 Index']
pos_adjust_1000 = pos_adjust_1000.sort_values(by='amount_diff_abs', ascending=False)

pos_adjust_1000['expanding_amount'] = pos_adjust_1000['amount_diff'].expanding().sum()
pos_adjust_1000['expanding_amount_adjust'] = pos_adjust_1000['expanding_amount'] + index_exposure.loc['CSI1000 Index']
pos_adjust_1000['expanding_amount_adjust_abs'] = pos_adjust_1000['expanding_amount_adjust'].abs()
pos_adjust_1000 = pos_adjust_1000.reset_index(drop=True)
pos_adjust_1000.to_excel('1000调整明细.xlsx')